# Map and Recude exploration

In [1]:
from langchain.chains import MapReduceDocumentsChain, ReduceDocumentsChain
from langchain.chains.combine_documents.stuff import StuffDocumentsChain
from langchain.chains.llm import LLMChain
from langchain.prompts import PromptTemplate
from langchain.llms import OpenAI
from langchain.text_splitter import CharacterTextSplitter

In [2]:
# Map prompt: Summarize each chunk
map_template = """The following is a chunk of text:
{text}
Provide a concise summary of this chunk:"""
map_prompt = PromptTemplate.from_template(map_template)

# Reduce prompt: Combine all summaries into one
reduce_template = """The following are summaries of different chunks of text:
{text}
Combine these summaries into one coherent final summary:"""
reduce_prompt = PromptTemplate.from_template(reduce_template)

In [18]:
from langchain_google_vertexai import (
    ChatVertexAI,
    HarmBlockThreshold,
    HarmCategory,
)

safety_settings = {
    HarmCategory.HARM_CATEGORY_UNSPECIFIED: HarmBlockThreshold.BLOCK_NONE,
    HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
    HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
    HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
    HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_NONE,
}

llm = ChatVertexAI(model="gemini-1.5-pro", temperature=0, safety_settings=safety_settings)

In [4]:
MAX_TOEKNS = 128000

In [5]:
# Map chain: Summarize each chunk
map_chain = LLMChain(llm=llm, prompt=map_prompt)

# Reduce chain: Combine summaries
reduce_chain = LLMChain(llm=llm, prompt=reduce_prompt)

# Combine documents using StuffDocumentsChain
combine_documents_chain = StuffDocumentsChain(
    llm_chain=reduce_chain, document_variable_name="text"
)

# Reduce documents chain
reduce_documents_chain = ReduceDocumentsChain(
    combine_documents_chain=combine_documents_chain,
    collapse_documents_chain=combine_documents_chain,
    token_max=MAX_TOEKNS,  # Adjust based on model token limits
)

# Map-reduce chain
map_reduce_chain = MapReduceDocumentsChain(
    llm_chain=map_chain,
    reduce_documents_chain=reduce_documents_chain,
    document_variable_name="text",
)

/tmp/ipykernel_209605/31706053.py:2: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  map_chain = LLMChain(llm=llm, prompt=map_prompt)
/tmp/ipykernel_209605/31706053.py:8: LangChainDeprecationWarning: This class is deprecated. Use the `create_stuff_documents_chain` constructor instead. See migration guide here: https://python.langchain.com/docs/versions/migrating_chains/stuff_docs_chain/
  combine_documents_chain = StuffDocumentsChain(
/tmp/ipykernel_209605/31706053.py:13: LangChainDeprecationWarning: This class is deprecated. Please see the migration guide here for a recommended replacement: https://python.langchain.com/docs/versions/migrating_chains/map_reduce_chain/
  reduce_documents_chain = ReduceDocumentsChain(
/tmp/ipykernel_209605/31706053.py:20: LangChainDeprecationWarning: This class is deprecated. Please see the migration guide here for a recommended

In [6]:
text_splitter = CharacterTextSplitter(
    chunk_size=MAX_TOEKNS / 5,  # Adjust based on your needs
    chunk_overlap=200,  # Overlap to maintain context
)

In [7]:
# Example text
text = "Your long input text goes here..."

# Split the text
docs = text_splitter.create_documents([text])

In [8]:
summary = map_reduce_chain.run(docs)
print("Final Summary:", summary)

/tmp/ipykernel_209605/1865348454.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  summary = map_reduce_chain.run(docs)


Final Summary: Please provide me with the summaries you want me to combine. I need the actual summaries to create a coherent final summary for you. 

For example, you could say:

"Here are the summaries:
* **Summary 1:** This article discusses the impact of climate change on coastal cities.
* **Summary 2:** This section explains the science behind rising sea levels.
* **Summary 3:** The author proposes solutions like building sea walls and reducing carbon emissions."

I will then combine these summaries into a single, coherent summary. 



In [9]:
from summarization import summarize_long_text

long_text = """
Patient John Doe visited City Hospital on 2023-10-01 for a routine check-up. Dr. Smith, a cardiologist, conducted the examination. The total cost was $200, covered by insurance. The patient was prescribed medication for high blood pressure and advised to return in 3 months.
    """

In [10]:
summary = summarize_long_text(long_text)
print("Final Summary:\n", summary)

Final Summary:
 ## Patient Summary: John Doe

**Date:** 2023-10-01

**Patient:** John Doe 

**Summary:**

Mr. Doe underwent a routine cardiology check-up with Dr. Smith at City Hospital.  He was prescribed medication for high blood pressure and advised to schedule a follow-up appointment in 3 months. 

**Healthcare Provider:** Dr. Smith, Cardiologist at City Hospital

**Financial Information:** The total cost of the visit was $200, which was covered by the patient's insurance. 



## Map and reduce parallel

In [97]:
import concurrent.futures
from langchain.chains import MapReduceDocumentsChain, ReduceDocumentsChain
from langchain.chains.combine_documents.stuff import StuffDocumentsChain
from langchain.chains.llm import LLMChain
from langchain.prompts import PromptTemplate
from langchain.text_splitter import CharacterTextSplitter
from langchain.docstore.document import Document

SUMMARIZATION_MAP_PROMPT = map_template
SUMMARIZATION_REDUCE_PROMPT = reduce_template

def summarize_long_text(long_text: str, llm) -> str:
    """
    Summarizes a long text using a map-reduce approach with LangChain and parallel execution.
    """
    # Initialize the LLM
    #llm = config.LLM_MODEL_SUMMARIZATION

    # Define the map and reduce prompts
    map_template = SUMMARIZATION_MAP_PROMPT
    map_prompt = PromptTemplate.from_template(map_template)

    reduce_template = SUMMARIZATION_REDUCE_PROMPT
    reduce_prompt = PromptTemplate.from_template(reduce_template)

    # Create the map and reduce chains
    map_chain = LLMChain(llm=llm, prompt=map_prompt)
    reduce_chain = LLMChain(llm=llm, prompt=reduce_prompt)

    # Combine documents using StuffDocumentsChain
    combine_documents_chain = StuffDocumentsChain(
        llm_chain=reduce_chain, document_variable_name="text"
    )

    # Reduce documents chain
    reduce_documents_chain = ReduceDocumentsChain(
        combine_documents_chain=combine_documents_chain,
        collapse_documents_chain=combine_documents_chain,
        token_max=128_000,  # Adjust based on model token limits
    )

    # Split the text into chunks
    text_splitter = CharacterTextSplitter(
        chunk_size=10,  # Adjust based on your model's token limit
        chunk_overlap=3,  # Overlap to maintain context
    )
    docs = text_splitter.create_documents([long_text])

    # Function to process a single document chunk
    def process_chunk(chunk):
        return map_chain.run([chunk])

    # Process chunks in parallel using ThreadPoolExecutor
    with concurrent.futures.ThreadPoolExecutor() as executor:
        # Submit all chunks for parallel processing
        future_to_chunk = {executor.submit(process_chunk, chunk): chunk for chunk in docs}
        summaries = []
        for future in concurrent.futures.as_completed(future_to_chunk):
            try:
                summary = future.result()
                summaries.append(summary)
            except Exception as e:
                print(f"Error processing chunk: {e}")

    # Combine the summaries into a single document
    combined_summary = " ".join(summaries)
    doc =  Document(page_content=combined_summary, metadata={"source": "local"})
    return reduce_documents_chain.run([doc])

In [98]:
long_text = """
Patient John Doe visited City Hospital on 2023-10-01 for a routine check-up. Dr. Smith, a cardiologist, conducted the examination. The total cost was $200, covered by insurance. The patient was prescribed medication for high blood pressure and advised to return in 3 months.
    """

In [99]:
final_summary = summarize_long_text(long_text, llm)

In [100]:
final_summary

'On October 1st, 2023, John Doe had a routine cardiology check-up with Dr. Smith at City Hospital. Dr. Smith prescribed him blood pressure medication and scheduled a follow-up appointment in three months. The visit cost $200 and was covered by insurance. \n'

In [86]:
from langchain.docstore.document import Document

doc =  Document(page_content=combined_summary, metadata={"source": "local"})


In [67]:
combined_summary

'John Doe had a routine check-up with cardiologist Dr. Smith at City Hospital on 2023-10-01. He was prescribed blood pressure medication and advised to return in 3 months. The visit cost $200 and was covered by insurance. \n'

In [68]:
# Combine documents using StuffDocumentsChain
combine_documents_chain = StuffDocumentsChain(
    llm_chain=reduce_chain, document_variable_name="text"
)

# Reduce documents chain
reduce_documents_chain = ReduceDocumentsChain(
    combine_documents_chain=combine_documents_chain,
    collapse_documents_chain=combine_documents_chain,
    token_max=128_000,  # Adjust based on model token limits
)

In [90]:
final_summary = reduce_documents_chain.run([doc])

In [91]:
final_summary

'On October 1st, 2023, John Doe had a routine cardiology check-up with Dr. Smith at City Hospital. Dr. Smith prescribed blood pressure medication and advised a follow-up appointment in three months. The visit cost $200 and was covered by insurance. \n'

## SUmmarization lang chain

In [78]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains.summarize import load_summarize_chain
from langchain.prompts import PromptTemplate
from langchain.llms import OpenAI
from concurrent.futures import ThreadPoolExecutor

# Initialize LLM
# Define text splitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=10)

# Input text
input_text = """
[Insert large text here, such as a long article or document]
"""

# Split the input text into chunks
#chunks = text_splitter.split_text(input_text)
chunks = text_splitter.create_documents([input_text])

# Define a function to summarize a single chunk
def summarize_chunk(chunk):
    prompt_template = """
    Summarize the following text:
    {text}
    """
    prompt = PromptTemplate(template=prompt_template, input_variables=["text"])
    chain = load_summarize_chain(llm, chain_type="stuff", prompt=prompt)
    return chain.run(chunk)

# Perform the map step in parallel
with ThreadPoolExecutor() as executor:
    partial_summaries = list(executor.map(summarize_chunk, chunks))

# Perform the reduce step (combine summaries)
final_summary_prompt = """
Combine the following partial summaries into a concise summary:
{summaries}
"""
prompt = PromptTemplate(template=final_summary_prompt, input_variables=["summaries"])
reduce_chain = load_summarize_chain(llm, chain_type="stuff", prompt=prompt)

AttributeError: 'tuple' object has no attribute 'page_content'

In [ ]:

# Combine partial summaries into a final summary
final_summary = reduce_chain.run("\n".join(partial_summaries))

print("Final Summary:")
print(final_summary)


In [83]:
type(chunks[0].page_content)

langchain_core.documents.base.Document

In [80]:
reduce_chain = LLMChain(llm=llm, prompt=reduce_prompt)

# Combine documents using StuffDocumentsChain
combine_documents_chain = StuffDocumentsChain(
    llm_chain=reduce_chain, document_variable_name="text"
)

# Reduce documents chain
reduce_documents_chain = ReduceDocumentsChain(
    combine_documents_chain=combine_documents_chain,
    collapse_documents_chain=combine_documents_chain,
    token_max=128_000,  # Adjust based on model token limits
)


In [84]:
reduce_documents_chain.run(chunks)

'Please provide me with the text summaries you want me to combine. I need the actual summaries to create a coherent final summary. \n\nFor example, you can provide the summaries in the following format:\n\n**Summary 1:** [Insert summary of the first chunk of text]\n**Summary 2:** [Insert summary of the second chunk of text]\n**Summary 3:** [Insert summary of the third chunk of text]\n...\n\nOnce you provide me with the summaries, I can help you combine them into a single, comprehensive summary. \n'

In [23]:
summary = summarize_long_text(long_text, l_lm)
print("Final Summary:\n", summary)

AttributeError: 'dict' object has no attribute 'page_content'